# Kaggle NB-05: Prompting & Decode Sweep

**Self-contained Kaggle notebook.** Run with: Kernel → Run All

## What this notebook does
1. Clones the project repo (`src/`) into `/kaggle/working/repo` and wires up `sys.path`
2. Loads the competition `train.csv` from `/kaggle/input/`
3. Normalises raw rows into the canonical JSONL format used by the eval harness
4. Creates stratified `validation_200` and `golden_20` splits inline (reproduces notebook 03)
5. Loads the Nemotron model via KaggleHub
6. Runs a **quick baseline eval** (20 examples) to establish a reference run directory
7. Runs a **prompting sweep** across `zero-shot-cot` / `few-shot-cot` x decode grid x seeds
8. Writes a ranked CSV and a findings markdown
9. Prints a summary table

## Key variable
Set `MAX_EXAMPLES_PER_RUN = 20` for a quick sanity check.  
Set `MAX_EXAMPLES_PER_RUN = 200` (or `None`) for the full validation sweep.

## Cell 1 — Install dependencies

In [ ]:
# Install any packages not pre-installed on Kaggle.
# transformers, torch, pandas are already present on Kaggle GPU kernels.
import subprocess, sys

def _pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *args], check=True)

# kagglehub ships on Kaggle kernels; ensure it is present.
_pip('kagglehub>=0.3')

print('Dependency check complete.')

## Cell 2 — Configuration

Change `MAX_EXAMPLES_PER_RUN` here.  
- `20`  -> quick sanity check (~5-10 min on GPU)
- `200` -> full validation sweep (several hours)

In [ ]:
# ── USER-CONFIGURABLE ────────────────────────────────────────────────────────
MAX_EXAMPLES_PER_RUN = 20   # set to 200 for full run, None for unlimited
MAX_NEW_TOKENS       = 256
SEED                 = 42
DATASET_VERSION      = 'kaggle-v1'
NORMALIZER_ID        = 'strip_v1'

# ── PATHS ────────────────────────────────────────────────────────────────────
from pathlib import Path

KAGGLE_INPUT    = Path('/kaggle/input')
WORKING_DIR     = Path('/kaggle/working')
REPO_DIR        = WORKING_DIR / 'repo'
OUTPUT_DIR      = WORKING_DIR / 'eval'
RUNS_DIR        = OUTPUT_DIR / 'runs'
BASELINE_RUN_ID = 'baseline-real-v1'

RUNS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Working dir         : {WORKING_DIR}')
print(f'Output dir          : {OUTPUT_DIR}')
print(f'Max examples per run: {MAX_EXAMPLES_PER_RUN}')

## Cell 3 — Wire up src/ from repo or inline fallback

Tries three strategies in order:
1. A dataset attachment mounted at `/kaggle/input/*/src`
2. A `git clone` of the public GitHub repo
3. Inline implementations written directly into `/kaggle/working/inline_src/`

In [ ]:
import os, sys, json, subprocess, re, time, random, hashlib, csv
from collections import Counter, defaultdict
from pathlib import Path

# ── Strategy 1: mounted dataset ────────────────────────────────────────────
_mounted_src = None
for _cand in KAGGLE_INPUT.glob('*/src'):
    if (_cand / 'contracts.py').exists() or (_cand / 'evaluation').is_dir():
        _mounted_src = _cand.parent
        break

if _mounted_src is not None:
    SRC_ROOT = _mounted_src
    print(f'Strategy 1: mounted src/ at {SRC_ROOT}')
else:
    # ── Strategy 2: git clone ────────────────────────────────────────────
    REPO_URL = 'https://github.com/DangT/CS4650-Nvidia-Nemotron-Challenge.git'
    if not REPO_DIR.exists():
        _r = subprocess.run(
            ['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)],
            capture_output=True, text=True,
        )
        if _r.returncode == 0:
            print(f'Strategy 2: cloned to {REPO_DIR}')
        else:
            print(f'Strategy 2 failed: {_r.stderr[:200]}')
            REPO_DIR.mkdir(parents=True, exist_ok=True)
    SRC_ROOT = REPO_DIR

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

# ── Probe import ─────────────────────────────────────────────────────────
_have_real_src = False
try:
    from src.contracts import EvalRecord, ReasoningExample
    from src.evaluation.normalization import normalize, available_normalizers
    from src.evaluation.scoring import score_exact_match, score_records
    from src.evaluation.config import make_run_config, EvalRunConfig
    from src.evaluation.runner import run_baseline_eval, PredictRequest, PredictResponse
    from src.evaluation.reporting import write_run_artifacts
    from src.evaluation.splits import SplitArtifactRow, write_split_jsonl
    from src.evaluation.golden_gate import evaluate_golden_gate, summarize_gate
    from src.evaluation.prompt_sweeps import (
        DEFAULT_STRATEGIES, DEFAULT_DECODE_GRID, DEFAULT_SEEDS, DEFAULT_BEST_OF_N,
        build_run_id, build_sparse_sweep_specs, build_best_of_n_specs,
        aggregate_sweep_results, majority_vote, stable_int_seed,
        render_findings_markdown, write_aggregate_csv, write_findings_markdown,
        AggregateSweepRow,
    )
    _have_real_src = True
    print('Real src/ loaded.')
except ImportError as _e:
    print(f'Real src/ not available ({_e}); will use inline fallback.')

## Cell 4 — Inline fallback implementations

If neither the mounted dataset nor the git clone succeeded, we define all
required classes and functions directly in this notebook.  These are exact
minimal reimplementations of the real `src/` modules.

In [ ]:
if not _have_real_src:
    # ── Contracts ────────────────────────────────────────────────────────
    from dataclasses import dataclass, field, asdict
    from statistics import mean, median
    from datetime import datetime, timezone
    from typing import Any

    @dataclass(frozen=True)
    class ReasoningExample:
        id: str; category: str; prompt: str; answer: str
        source: str; split: str; metadata: dict = field(default_factory=dict)

    @dataclass(frozen=True)
    class EvalRecord:
        run_id: str; example_id: str; model_id: str; prompt_template_id: str
        normalizer_id: str; category: str; split: str; gold: str
        prediction: str; normalized_prediction: str; correct: bool
        latency_ms: float; tokens_in: int; tokens_out: int; seed: int
        decode_config: dict = field(default_factory=dict)

    @dataclass(frozen=True)
    class SplitArtifactRow:
        example_id: str; category: str; prompt: str; gold: str
        source: str; split: str; dataset_version: str
        selection_seed: int; selection_rule: str; selection_reason: str

    # ── Normalization ────────────────────────────────────────────────────
    _NORM_REG = {
        'exact_v1':       lambda r, c: r,
        'strip_v1':       lambda r, c: r.strip(),
        'collapse_ws_v1': lambda r, c: ' '.join(r.split()),
        'last_line_v1':   lambda r, c: next(
            (l.strip() for l in reversed(r.splitlines()) if l.strip()), ''),
    }
    def normalize(raw, *, normalizer_id, category=None):
        return _NORM_REG[normalizer_id](raw, category)
    def available_normalizers():
        return sorted(_NORM_REG)

    # ── Scoring ──────────────────────────────────────────────────────────
    def score_exact_match(norm, gold):
        return bool(norm) and norm == gold

    def _pct(vals, p):
        if not vals: return 0.0
        s = sorted(vals); k = (len(s)-1)*(p/100)
        lo = int(k); hi = min(lo+1, len(s)-1)
        return float(s[lo] + (s[hi]-s[lo])*(k-lo))

    def score_records(records):
        total = len(records); correct = sum(1 for r in records if r.correct)
        per_cat = {}
        for rec in records:
            b = per_cat.setdefault(rec.category, {'total':0,'correct':0,'accuracy':0.0})
            b['total'] += 1
            if rec.correct: b['correct'] += 1
        for s in per_cat.values():
            s['accuracy'] = s['correct']/s['total'] if s['total'] else 0.0
        lats = [r.latency_ms for r in records]
        return dict(
            total=total, correct=correct,
            accuracy=(correct/total if total else 0.0),
            per_category_accuracy={c: dict(per_cat[c]) for c in sorted(per_cat)},
            latency_ms={'mean': float(mean(lats)) if lats else 0.0,
                        'p50':  float(median(lats)) if lats else 0.0,
                        'p95':  _pct(lats, 95)},
            tokens_in_total=sum(r.tokens_in for r in records),
            tokens_out_total=sum(r.tokens_out for r in records),
            run_id=records[0].run_id if records else None,
            model_id=records[0].model_id if records else None,
            prompt_template_id=records[0].prompt_template_id if records else None,
            normalizer_id=records[0].normalizer_id if records else None,
            seed=records[0].seed if records else None,
            split=records[0].split if records else None,
            normalizer_ids=sorted({r.normalizer_id for r in records}),
            run_ids=sorted({r.run_id for r in records}),
        )

    # ── Config ───────────────────────────────────────────────────────────
    @dataclass(frozen=True)
    class EvalRunConfig:
        run_id: str; model_id: str; prompt_template_id: str
        normalizer_id: str; seed: int; split: str; dataset_version: str
        decode_config: dict = field(default_factory=dict)
        created_at: str = ''

    def make_run_config(*, run_id, model_id, prompt_template_id, normalizer_id,
                        seed, split, dataset_version, decode_config=None, created_at=None):
        stamp = created_at or datetime.now(timezone.utc).isoformat(timespec='seconds')
        return EvalRunConfig(run_id=run_id, model_id=model_id,
                             prompt_template_id=prompt_template_id,
                             normalizer_id=normalizer_id, seed=seed,
                             split=split, dataset_version=dataset_version,
                             decode_config=dict(decode_config or {}),
                             created_at=stamp)

    # ── Splits I/O ───────────────────────────────────────────────────────
    def write_split_jsonl(rows, path):
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        with open(path, 'w', encoding='utf-8') as fh:
            for row in rows:
                fh.write(json.dumps(asdict(row), ensure_ascii=False) + '\n')
        return len(rows)

    def read_split_jsonl(path):
        rows = []
        with open(path, 'r', encoding='utf-8') as fh:
            for line in fh:
                line = line.strip()
                if line: rows.append(SplitArtifactRow(**json.loads(line)))
        return rows

    # ── Reporting helpers ─────────────────────────────────────────────────
    @dataclass(frozen=True)
    class RawPrediction:
        run_id: str; example_id: str; model_id: str; prompt_template_id: str
        seed: int; prediction: str; latency_ms: float; tokens_in: int; tokens_out: int

    def _write_jsonl(rows, path):
        with open(path, 'w', encoding='utf-8') as fh:
            for r in rows: fh.write(json.dumps(asdict(r), ensure_ascii=False) + '\n')

    def write_run_config_file(config, path):
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        with open(path, 'w', encoding='utf-8') as fh:
            json.dump(asdict(config), fh, indent=2, sort_keys=True)
            fh.write('\n')

    def write_run_artifacts(*, output_dir, config, raw_predictions, records):
        od = Path(output_dir); od.mkdir(parents=True, exist_ok=True)
        pp = od / 'predictions.jsonl'; ep = od / 'eval_records.jsonl'
        cp = od / 'run_config.json'; sp = od / 'summary.json'
        _write_jsonl(raw_predictions, pp)
        _write_jsonl(records, ep)
        write_run_config_file(config, cp)
        sm = score_records(records)
        with open(sp, 'w', encoding='utf-8') as fh:
            json.dump(sm, fh, indent=2, sort_keys=True)
            fh.write('\n')
        return {'predictions': pp, 'eval_records': ep, 'run_config': cp, 'summary': sp}

    # ── Runner ────────────────────────────────────────────────────────────
    @dataclass(frozen=True)
    class PredictRequest:
        example_id: str; category: str; prompt: str; seed: int

    @dataclass(frozen=True)
    class PredictResponse:
        prediction: str; latency_ms: float; tokens_in: int; tokens_out: int

    @dataclass(frozen=True)
    class RunResult:
        records: list; raw_predictions: list; summary: dict
        artifact_paths: dict = field(default_factory=dict)

    def run_baseline_eval(*, split_rows, config, predictor, output_dir=None):
        raws, recs = [], []
        for row in split_rows:
            req  = PredictRequest(example_id=row.example_id, category=row.category,
                                   prompt=row.prompt, seed=config.seed)
            resp = predictor(req)
            norm = normalize(resp.prediction, normalizer_id=config.normalizer_id,
                             category=row.category)
            ok   = score_exact_match(norm, row.gold)
            raws.append(RawPrediction(
                run_id=config.run_id, example_id=row.example_id,
                model_id=config.model_id, prompt_template_id=config.prompt_template_id,
                seed=config.seed, prediction=resp.prediction,
                latency_ms=float(resp.latency_ms),
                tokens_in=resp.tokens_in, tokens_out=resp.tokens_out))
            recs.append(EvalRecord(
                run_id=config.run_id, example_id=row.example_id,
                model_id=config.model_id, prompt_template_id=config.prompt_template_id,
                normalizer_id=config.normalizer_id, category=row.category,
                split=row.split, gold=row.gold, prediction=resp.prediction,
                normalized_prediction=norm, correct=ok,
                latency_ms=float(resp.latency_ms),
                tokens_in=resp.tokens_in, tokens_out=resp.tokens_out,
                seed=config.seed, decode_config=dict(config.decode_config)))
        summary = score_records(recs)
        paths = {}
        if output_dir is not None:
            paths = write_run_artifacts(output_dir=output_dir, config=config,
                                        raw_predictions=raws, records=recs)
        return RunResult(records=recs, raw_predictions=raws, summary=summary, artifact_paths=paths)

    # ── Golden gate ───────────────────────────────────────────────────────
    @dataclass(frozen=True)
    class GoldenGateResult:
        passed: bool; total: int; misses: list = field(default_factory=list)

    def evaluate_golden_gate(records, golden):
        by_id = {}
        for rec in records:
            by_id.setdefault(rec.example_id, []).append(rec)
        misses = []
        for row in golden:
            recs = by_id.get(row.example_id)
            if not recs:
                misses.append({'example_id': row.example_id, 'category': row.category,
                               'gold': row.gold, 'normalized_prediction': '',
                               'reason': 'missing'})
            elif not all(r.correct for r in recs):
                misses.append({'example_id': row.example_id, 'category': row.category,
                               'gold': row.gold,
                               'normalized_prediction': recs[0].normalized_prediction,
                               'reason': 'incorrect'})
        return GoldenGateResult(passed=len(misses)==0, total=len(golden), misses=misses)

    def summarize_gate(result):
        verdict = 'PASS' if result.passed else 'FAIL'
        n_pass = result.total - len(result.misses)
        lines = [f'Golden gate: {verdict} (passed={n_pass}/{result.total})']
        for m in result.misses:
            lines.append(f'  - {m["example_id"]} [{m["category"]}] reason={m["reason"]}')
        return '\n'.join(lines)

    # ── Prompt sweeps ─────────────────────────────────────────────────────
    DEFAULT_STRATEGIES  = ('zero-shot-cot', 'few-shot-cot')
    DEFAULT_DECODE_GRID = ((0.6, 0.9), (0.6, 0.95), (1.0, 0.9), (1.0, 0.95))
    DEFAULT_SEEDS       = (11, 23, 37)
    DEFAULT_BEST_OF_N   = (8, 32)

    def _float_tok(v): return f'{v:.2f}'.rstrip('0').rstrip('.').replace('.', 'p')
    def _slug(v): return v.strip().lower().replace('_','-').replace(' ','-')

    def build_run_id(*, date_stamp, strategy, temperature, top_p, seed, best_of_n=1):
        parts = ['prompt-sweep', date_stamp, _slug(strategy),
                 f't{_float_tok(temperature)}', f'p{_float_tok(top_p)}', f's{seed}']
        if best_of_n > 1: parts.append(f'bon{best_of_n}')
        return '-'.join(parts)

    @dataclass(frozen=True)
    class SweepRunSpec:
        run_id: str; strategy: str; temperature: float; top_p: float
        seed: int; best_of_n: int = 1

    def build_sparse_sweep_specs(*, date_stamp, strategies=DEFAULT_STRATEGIES,
                                  decode_grid=DEFAULT_DECODE_GRID, seeds=DEFAULT_SEEDS):
        specs = []
        for strategy in strategies:
            for temp, top_p in decode_grid:
                for seed in seeds:
                    specs.append(SweepRunSpec(
                        run_id=build_run_id(date_stamp=date_stamp, strategy=strategy,
                                            temperature=temp, top_p=top_p, seed=seed),
                        strategy=strategy, temperature=temp, top_p=top_p, seed=seed))
        return specs

    def build_best_of_n_specs(*, date_stamp, strategy, temperature, top_p,
                               seeds=DEFAULT_SEEDS, best_of_n_values=DEFAULT_BEST_OF_N):
        specs = []
        for bon in best_of_n_values:
            for seed in seeds:
                specs.append(SweepRunSpec(
                    run_id=build_run_id(date_stamp=date_stamp, strategy=strategy,
                                        temperature=temperature, top_p=top_p,
                                        seed=seed, best_of_n=bon),
                    strategy=strategy, temperature=temperature, top_p=top_p,
                    seed=seed, best_of_n=bon))
        return specs

    def stable_int_seed(*parts):
        d = hashlib.sha256('::'.join(str(p) for p in parts).encode()).digest()
        return int.from_bytes(d[:8], 'big') % (2**31)

    def majority_vote(predictions):
        if not predictions: raise ValueError('empty')
        c = Counter(predictions); best = max(c.values())
        return sorted(p for p, n in c.items() if n == best)[0]

    @dataclass(frozen=True)
    class AggregateSweepRow:
        strategy: str; temperature: float; top_p: float; best_of_n: int
        run_count: int; accuracy_mean: float; accuracy_std: float
        latency_seconds_mean: float; latency_seconds_std: float
        delta_vs_baseline: float; significant: bool
        run_ids: tuple; seeds: tuple

        def as_csv_row(self):
            return dict(strategy=self.strategy, temperature=self.temperature,
                        top_p=self.top_p, best_of_n=self.best_of_n,
                        run_count=self.run_count, accuracy_mean=self.accuracy_mean,
                        accuracy_std=self.accuracy_std,
                        latency_seconds_mean=self.latency_seconds_mean,
                        latency_seconds_std=self.latency_seconds_std,
                        delta_vs_baseline=self.delta_vs_baseline,
                        significant=self.significant,
                        run_ids='|'.join(self.run_ids),
                        seeds='|'.join(str(s) for s in self.seeds))

    def aggregate_sweep_results(rows, *, baseline_accuracy):
        from statistics import stdev
        buckets = {}
        for row in rows:
            key = (str(row['strategy']), float(row['temperature']),
                   float(row['top_p']), int(row.get('best_of_n', 1)))
            buckets.setdefault(key, []).append(row)
        out = []
        for (strat, temp, top_p, bon), bucket in buckets.items():
            accs = [float(r['accuracy']) for r in bucket]
            lats = [float(r['elapsed_seconds']) for r in bucket]
            am   = mean(accs); astd = stdev(accs) if len(accs) > 1 else 0.0
            lm   = mean(lats); lstd = stdev(lats) if len(lats) > 1 else 0.0
            delta = am - baseline_accuracy
            out.append(AggregateSweepRow(
                strategy=strat, temperature=temp, top_p=top_p, best_of_n=bon,
                run_count=len(bucket), accuracy_mean=am, accuracy_std=astd,
                latency_seconds_mean=lm, latency_seconds_std=lstd,
                delta_vs_baseline=delta, significant=delta > 2*astd,
                run_ids=tuple(sorted(str(r['run_id']) for r in bucket)),
                seeds=tuple(sorted(int(r['seed']) for r in bucket))))
        return sorted(out, key=lambda r: (-r.accuracy_mean, r.accuracy_std,
                                          r.latency_seconds_mean, r.strategy))

    def write_aggregate_csv(rows, path):
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        fields = ['strategy','temperature','top_p','best_of_n','run_count',
                  'accuracy_mean','accuracy_std','latency_seconds_mean',
                  'latency_seconds_std','delta_vs_baseline','significant',
                  'run_ids','seeds']
        with open(path, 'w', newline='', encoding='utf-8') as fh:
            w = csv.DictWriter(fh, fieldnames=fields)
            w.writeheader()
            for row in rows: w.writerow(row.as_csv_row())
        return Path(path)

    def render_findings_markdown(*, date_stamp, baseline, aggregate_rows, csv_path,
                                  golden_gate_passed, golden_summary, promoted_run_id, notes=()):
        # baseline may be a dict or an object; handle both
        _run_id  = baseline.get('run_id', '?') if isinstance(baseline, dict) else str(baseline)
        _acc     = baseline.get('accuracy', 0.0) if isinstance(baseline, dict) else 0.0
        best     = aggregate_rows[0]
        lines    = ['# Prompting Findings', '', f'Date: `{date_stamp}`', '',
                    '## Baseline', '', f'- Baseline run id: `{_run_id}`',
                    f'- Baseline accuracy: `{_acc:.4f}`', '',
                    '## Ranked Configurations', '',
                    '| Rank | Strategy | Temp | Top-p | BoN | Mean Acc | Std | Delta | Sig |',
                    '|---|---|---:|---:|---:|---:|---:|---:|---:|']
        for rank, row in enumerate(aggregate_rows, 1):
            lines.append(f'| {rank} | `{row.strategy}` | {row.temperature:.2f} | '
                         f'{row.top_p:.2f} | {row.best_of_n} | {row.accuracy_mean:.4f} | '
                         f'{row.accuracy_std:.4f} | {row.delta_vs_baseline:+.4f} | {row.significant} |')
        lines += ['', '## Promotion Decision', '',
                  f'- Promoted: `{promoted_run_id}`',
                  f'- Golden gate passed: `{golden_gate_passed}`', '',
                  '## Golden Regression Check', '', '```text',
                  golden_summary.strip(), '```', '']
        note_list = list(notes)
        if note_list:
            lines += ['## Notes', '']
            for n in note_list: lines.append(f'- {n}')
        return '\n'.join(lines) + '\n'

    def write_findings_markdown(content, path):
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        Path(path).write_text(content, encoding='utf-8')
        return Path(path)

    print('Inline fallback implementations loaded.')

print(f'Available normalizers: {available_normalizers()}')

## Cell 5 — Load competition data from `/kaggle/input/`

Expected: `train.csv` with columns `id`, `prompt`, `answer` (and optionally `category`).  
If the CSV is in a subdirectory the notebook finds it via glob.

In [ ]:
import pandas as pd

# Find train.csv — may be nested inside a competition-named subdirectory.
_candidates = list(KAGGLE_INPUT.rglob('train.csv'))
if not _candidates:
    raise FileNotFoundError(
        f'train.csv not found under {KAGGLE_INPUT}.\n'
        'Add the competition dataset:\n'
        '  Notebook → Data → Add Data → Competition Data → '
        'nvidia-arc-model-reasoning-challenge'
    )

TRAIN_CSV = _candidates[0]
print(f'Loading: {TRAIN_CSV}')
train_df = pd.read_csv(TRAIN_CSV)
print(f'Shape  : {train_df.shape}')
print(f'Columns: {list(train_df.columns)}')
train_df.head(3)

## Cell 6 — Normalise raw CSV rows to canonical JSONL format

If `category` is absent we infer it from prompt keywords.  
Output fields: `example_id`, `category`, `prompt`, `gold`, `source`, `split`, `dataset_version`, `metadata`.

In [ ]:
CATEGORIES = [
    'bit_manipulation', 'equation_symbolic', 'numeral_system',
    'physics_gravity',  'text_cipher',        'unit_conversion',
]

_KEYWORDS = {
    'bit_manipulation': ['bit', 'binary', 'xor', 'and', 'or', 'not', 'shift', 'rotation', 'bitwise'],
    'equation_symbolic': ['factor', 'polynomial', 'symbolic', 'expand', 'simplify', 'algebra'],
    'numeral_system': ['hexadecimal', 'octal', 'base', 'numeral', 'convert.*to.*binary', 'binary.*to.*decimal'],
    'physics_gravity': ['gravity', 'gravit', 'falls', 'acceleration', 'g=9', 'free fall', 'distance.*fall'],
    'text_cipher': ['cipher', 'rot13', 'rot-13', 'caesar', 'decode.*letter', 'encode.*letter'],
    'unit_conversion': ['convert', 'km/h', 'm/s', 'celsius', 'fahrenheit', 'miles', 'kilogram', 'unit'],
}

def infer_category(prompt_text: str) -> str:
    text = prompt_text.lower()
    scores = {}
    for cat, kws in _KEYWORDS.items():
        scores[cat] = sum(1 for kw in kws if re.search(kw, text))
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else 'unknown'

def _norm_row(row: pd.Series) -> dict:
    _id     = str(row.get('id', row.name))
    _prompt = str(row.get('prompt', row.get('question', '')))
    _gold   = str(row.get('answer', row.get('gold', '')))
    if 'category' in row.index and pd.notna(row['category']):
        _cat = str(row['category'])
    else:
        _cat = infer_category(_prompt)
    return dict(example_id=_id, category=_cat, prompt=_prompt, gold=_gold,
                source='kaggle-official', split='train',
                dataset_version=DATASET_VERSION, metadata={})

normalized_records = [_norm_row(row) for _, row in train_df.iterrows()]

NORMALIZED_JSONL = OUTPUT_DIR / 'train_normalized.jsonl'
NORMALIZED_JSONL.parent.mkdir(parents=True, exist_ok=True)
with NORMALIZED_JSONL.open('w', encoding='utf-8') as fh:
    for rec in normalized_records:
        fh.write(json.dumps(rec, ensure_ascii=False) + '\n')

print(f'Normalized {len(normalized_records)} records -> {NORMALIZED_JSONL}')
cat_counts = Counter(r['category'] for r in normalized_records)
print('\nCategory distribution:')
for cat, n in sorted(cat_counts.items()):
    print(f'  {cat:<25}: {n}')

## Cell 7 — Create validation_200 and golden_20 splits

Reproduces the notebook 03 logic inline: stratified by category, seed=42, no overlap.

In [ ]:
SPLIT_SEED = 42

by_cat = defaultdict(list)
for rec in normalized_records:
    by_cat[rec['category']].append(rec)

categories = sorted(by_cat.keys())
print(f'Categories: {categories}')

rng = random.Random(SPLIT_SEED)

# Shuffle each category pool deterministically
shuffled = {cat: rng.sample(by_cat[cat], len(by_cat[cat])) for cat in categories}

# ── Golden 20 (first claim) ────────────────────────────────────────────────
N_GOLDEN         = 20
per_cat_g        = N_GOLDEN // len(categories)
remainder_g      = N_GOLDEN % len(categories)
golden_records   = []
for i, cat in enumerate(categories):
    n = per_cat_g + (1 if i < remainder_g else 0)
    golden_records.extend(shuffled[cat][:n])

golden_ids = {r['example_id'] for r in golden_records}

# ── Validation 200 (from remaining rows) ──────────────────────────────────
N_VAL       = 200
rem_by_cat  = defaultdict(list)
for rec in normalized_records:
    if rec['example_id'] not in golden_ids:
        rem_by_cat[rec['category']].append(rec)

for cat in categories:
    rng.shuffle(rem_by_cat[cat])

per_cat_v   = N_VAL // len(categories)
remainder_v = N_VAL % len(categories)
val_records = []
for i, cat in enumerate(categories):
    n = per_cat_v + (1 if i < remainder_v else 0)
    val_records.extend(rem_by_cat[cat][:n])

rng.shuffle(val_records)

# Cap for quick runs
if MAX_EXAMPLES_PER_RUN is not None:
    print(f'Capping val split to {MAX_EXAMPLES_PER_RUN} (MAX_EXAMPLES_PER_RUN={MAX_EXAMPLES_PER_RUN})')
    val_records = val_records[:MAX_EXAMPLES_PER_RUN]

print(f'\nValidation split: {len(val_records)} rows')
for cat in categories:
    print(f'  {cat:<25}: {sum(1 for r in val_records if r["category"]==cat)}')
print(f'\nGolden split: {len(golden_records)} rows')

overlap = {r['example_id'] for r in golden_records} & {r['example_id'] for r in val_records}
assert len(overlap) == 0, f'Overlap detected: {overlap}'
print(f'\nOverlap check: 0 (OK)')

## Cell 8 — Convert dicts to SplitArtifactRow and write JSONL

In [ ]:
def _to_row(rec: dict, split_label: str) -> SplitArtifactRow:
    return SplitArtifactRow(
        example_id=rec['example_id'], category=rec['category'],
        prompt=rec['prompt'], gold=rec['gold'],
        source=rec.get('source', 'kaggle-official'),
        split=split_label, dataset_version=DATASET_VERSION,
        selection_seed=SPLIT_SEED,
        selection_rule='stratified-by-category',
        selection_reason=f'{split_label}:stratified-by-category; seed={SPLIT_SEED}',
    )

val_rows    = [_to_row(r, 'val')    for r in val_records]
golden_rows = [_to_row(r, 'golden') for r in golden_records]

VAL_JSONL    = OUTPUT_DIR / 'validation_200.jsonl'
GOLDEN_JSONL = OUTPUT_DIR / 'golden_20.jsonl'
write_split_jsonl(val_rows,    VAL_JSONL)
write_split_jsonl(golden_rows, GOLDEN_JSONL)

DATASET_VERSION_TAG = val_rows[0].dataset_version

print(f'Wrote {len(val_rows)} val rows    -> {VAL_JSONL}')
print(f'Wrote {len(golden_rows)} golden rows -> {GOLDEN_JSONL}')

## Cell 9 — Load the Nemotron model via KaggleHub

KaggleHub path: `metric/nemotron-3-nano-30b-a3b-bf16/transformers/default`  
(verified in `docs/architecture/COMPETITION.md`)

In [ ]:
import torch
import kagglehub
from transformers import AutoModelForCausalLM, AutoTokenizer

# Device check
cuda_ok = torch.cuda.is_available()
print('=' * 52)
print('Device summary')
print('=' * 52)
print(f'  CUDA available  : {cuda_ok}')
if cuda_ok:
    props = torch.cuda.get_device_properties(0)
    print(f'  GPU count       : {torch.cuda.device_count()}')
    print(f'  GPU name        : {props.name}')
    print(f'  GPU VRAM        : {props.total_memory / 1024**3:.1f} GB')
    print(f'  CUDA version    : {torch.version.cuda}')
print(f'  PyTorch version : {torch.__version__}')
print('=' * 52)

if not cuda_ok:
    raise RuntimeError(
        'A CUDA GPU is required.\n'
        'Enable: Notebook settings -> Accelerator -> GPU T4 x2 or P100.'
    )

DEVICE = torch.device('cuda')

# Load model
KAGGLE_MODEL_HANDLE = 'metric/nemotron-3-nano-30b-a3b-bf16/transformers/default'
MODEL_ID = KAGGLE_MODEL_HANDLE

print(f'\nDownloading model: {KAGGLE_MODEL_HANDLE}')
_t0 = time.perf_counter()
model_path = kagglehub.model_download(KAGGLE_MODEL_HANDLE)
print(f'Model path  : {model_path}')
print(f'Download    : {time.perf_counter()-_t0:.1f}s')

print('\nLoading tokenizer ...')
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)

print('Loading model (bfloat16, device_map=auto) ...')
torch.set_grad_enabled(False)
_t0 = time.perf_counter()
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
model.eval()
print(f'Model loaded: {time.perf_counter()-_t0:.1f}s')

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

INPUT_DEVICE = model.get_input_embeddings().weight.device
print(f'\nModel input device : {INPUT_DEVICE}')
print(f'Pad token id       : {tokenizer.pad_token_id}')

## Cell 10 — Prompt strategies and answer extraction

Two strategies:
- **zero-shot-cot**: reason step-by-step, then emit final answer in `\boxed{}`
- **few-shot-cot**: three worked examples prepended before the target problem

Extractor: prefers the last `\boxed{}` match; falls back to last non-empty line.

In [ ]:
FEW_SHOT_EXAMPLES = [
    {'question': 'Convert decimal 10 to binary.',
     'reasoning': '10 in decimal is 8 + 2, so the binary representation is 1010.',
     'answer': '1010'},
    {'question': 'Solve for x: 3x - 6 = 9.',
     'reasoning': 'Add 6 to both sides: 3x = 15, then divide by 3.',
     'answer': '5'},
    {'question': 'Decode ROT13: uryyb',
     'reasoning': 'ROT13 shifts each letter by 13 places, so uryyb becomes hello.',
     'answer': 'hello'},
]

SYSTEM_PROMPT = (
    'You solve short reasoning tasks. Work carefully. '
    'Always place your final answer inside \\boxed{...} on the last line. '
    'Do not include any other content after the \\boxed{} answer.'
)


def render_prompt(problem_prompt: str, *, strategy: str) -> str:
    if strategy == 'zero-shot-cot':
        return (
            'Solve the following task. Think step by step, then put only the '
            'final answer inside \\boxed{...} on the last line.\n\n'
            f'Task:\n{problem_prompt}'
        )
    if strategy == 'few-shot-cot':
        blocks = []
        for i, ex in enumerate(FEW_SHOT_EXAMPLES, start=1):
            blocks.append(
                f'Example {i}\nTask: {ex["question"]}\n'
                f'Reasoning: {ex["reasoning"]}\n'
                f'Answer: \\boxed{{{ex["answer"]}}}'
            )
        blocks.append(
            'Now solve the next task. Think step by step, then put only the '
            f'final answer inside \\boxed{{...}} on the last line.\n\nTask:\n{problem_prompt}'
        )
        return '\n\n'.join(blocks)
    raise ValueError(f'Unknown strategy: {strategy!r}')


def render_model_input(tokenizer, *, strategy: str, problem_prompt: str) -> str:
    user_text = render_prompt(problem_prompt, strategy=strategy)
    messages  = [{'role': 'system', 'content': SYSTEM_PROMPT},
                 {'role': 'user',   'content': user_text}]
    if hasattr(tokenizer, 'apply_chat_template'):
        try:
            return tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            pass
    return SYSTEM_PROMPT + '\n\n' + user_text + '\n\nAnswer:\n'


_BOXED_RE = re.compile(r'\\boxed\{([^}]*)\}')


def extract_final_answer(raw_text: str) -> str:
    matches = _BOXED_RE.findall(raw_text)
    if matches:
        return matches[-1].strip()
    lines = [l.strip() for l in raw_text.splitlines() if l.strip()]
    if not lines: return ''
    last = lines[-1]
    for prefix in ('final answer:', 'answer:'):
        if last.lower().startswith(prefix):
            return last.split(':', 1)[1].strip()
    return last


# Sanity checks
assert extract_final_answer('The result is \\boxed{42}.') == '42'
assert extract_final_answer('Thinking...\nFinal answer: 42') == '42'
print('Prompt strategies and extractor ready.')

## Cell 11 — Predictor factory

In [ ]:
def make_predictor(*, run_id: str, strategy: str, temperature: float,
                   top_p: float, best_of_n: int = 1):
    """Return a stateless PredictRequest -> PredictResponse callable."""
    if best_of_n < 1:
        raise ValueError(f'best_of_n must be >= 1, got {best_of_n}')

    def predictor(request: PredictRequest) -> PredictResponse:
        rendered = render_model_input(
            tokenizer, strategy=strategy, problem_prompt=request.prompt)
        encoded = tokenizer(rendered, return_tensors='pt')
        encoded = {k: v.to(INPUT_DEVICE) for k, v in encoded.items()}
        prompt_len = int(encoded['input_ids'].shape[1])

        gen_seed  = stable_int_seed(run_id, request.example_id, request.seed, best_of_n)
        generator = torch.Generator(device=str(INPUT_DEVICE))
        generator.manual_seed(gen_seed)

        _t = time.perf_counter()
        with torch.inference_mode():
            outputs = model.generate(
                **encoded,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                temperature=temperature,
                top_p=top_p,
                num_return_sequences=best_of_n,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                generator=generator,
            )
        elapsed_ms = (time.perf_counter() - _t) * 1000.0

        completion_ids = outputs[:, prompt_len:]
        decoded_texts  = tokenizer.batch_decode(completion_ids, skip_special_tokens=True)
        answers   = [extract_final_answer(txt) for txt in decoded_texts]
        prediction = majority_vote(answers) if len(answers) > 1 else answers[0]

        if tokenizer.pad_token_id is not None:
            tokens_out = int(sum(
                (seq != tokenizer.pad_token_id).sum().item() for seq in completion_ids))
        else:
            tokens_out = int(sum(seq.numel() for seq in completion_ids))

        return PredictResponse(
            prediction=prediction,
            latency_ms=elapsed_ms,
            tokens_in=prompt_len * best_of_n,
            tokens_out=tokens_out,
        )

    return predictor


print('Predictor factory ready.')

## Cell 12 — Quick baseline eval (greedy, no CoT, no `\boxed{}` instruction)

This is the "no prompting" reference that historically scores ~0% because the model
does not output `\boxed{}` without being asked.

In [ ]:
from datetime import datetime, timezone

DATE_STAMP = datetime.now(timezone.utc).date().isoformat()
print(f'Date stamp: {DATE_STAMP}')


def _greedy_predictor(request: PredictRequest) -> PredictResponse:
    """Minimal baseline: plain prompt, greedy decode, last-line extraction."""
    prompt_text = f'Answer the following task concisely:\n\n{request.prompt}\n\nAnswer: '
    encoded = tokenizer(prompt_text, return_tensors='pt')
    encoded = {k: v.to(INPUT_DEVICE) for k, v in encoded.items()}
    prompt_len = int(encoded['input_ids'].shape[1])

    _t = time.perf_counter()
    with torch.inference_mode():
        outputs = model.generate(
            **encoded,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False, temperature=1.0, top_p=1.0,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    elapsed_ms = (time.perf_counter() - _t) * 1000.0

    completion = outputs[:, prompt_len:]
    decoded    = tokenizer.decode(completion[0], skip_special_tokens=True)
    lines      = [l.strip() for l in decoded.splitlines() if l.strip()]
    prediction = lines[-1] if lines else ''

    tokens_out = int((completion[0] != tokenizer.pad_token_id).sum().item()) \
                 if tokenizer.pad_token_id is not None else completion[0].numel()

    return PredictResponse(prediction=prediction, latency_ms=elapsed_ms,
                           tokens_in=prompt_len, tokens_out=tokens_out)


baseline_config = make_run_config(
    run_id=BASELINE_RUN_ID, model_id=MODEL_ID,
    prompt_template_id='greedy-no-cot-v1', normalizer_id=NORMALIZER_ID,
    seed=SEED, split='val', dataset_version=DATASET_VERSION_TAG,
    decode_config={'temperature': 1.0, 'top_p': 1.0,
                   'max_new_tokens': MAX_NEW_TOKENS,
                   'do_sample': False, 'strategy': 'greedy-no-cot'},
)

BASELINE_RUN_DIR = RUNS_DIR / BASELINE_RUN_ID
BASELINE_RUN_DIR.mkdir(parents=True, exist_ok=True)

print(f'Running baseline on {len(val_rows)} examples ...')
_t0 = time.perf_counter()
baseline_result = run_baseline_eval(
    split_rows=val_rows, config=baseline_config,
    predictor=_greedy_predictor, output_dir=BASELINE_RUN_DIR,
)
BASELINE_ACCURACY = float(baseline_result.summary['accuracy'])

print()
print('=' * 52)
print('  BASELINE RESULT')
print('=' * 52)
print(f'  Accuracy : {BASELINE_ACCURACY:.4f}  ({BASELINE_ACCURACY:.1%})')
print(f'  Correct  : {baseline_result.summary["correct"]} / {baseline_result.summary["total"]}')
print(f'  Elapsed  : {time.perf_counter()-_t0:.1f}s')
print(f'  Run dir  : {BASELINE_RUN_DIR}')

## Cell 13 — Sparse prompt/decode sweep

`2 strategies x 4 decode configs x 3 seeds = 24 seeded runs`

With `MAX_EXAMPLES_PER_RUN=20` each run takes ~5-10 min on a T4 GPU.

In [ ]:
sparse_specs = build_sparse_sweep_specs(
    date_stamp=DATE_STAMP,
    strategies=DEFAULT_STRATEGIES,
    decode_grid=DEFAULT_DECODE_GRID,
    seeds=DEFAULT_SEEDS,
)
print(f'Sparse sweep: {len(sparse_specs)} runs')
print(f'  Strategies : {list(DEFAULT_STRATEGIES)}')
print(f'  Decode grid: {list(DEFAULT_DECODE_GRID)}')
print(f'  Seeds      : {list(DEFAULT_SEEDS)}')

sparse_rows = []
for i, spec in enumerate(sparse_specs, start=1):
    print(f'[{i:02d}/{len(sparse_specs):02d}] {spec.run_id}')
    predictor  = make_predictor(
        run_id=spec.run_id, strategy=spec.strategy,
        temperature=spec.temperature, top_p=spec.top_p, best_of_n=spec.best_of_n)
    run_config = make_run_config(
        run_id=spec.run_id, model_id=MODEL_ID,
        prompt_template_id=spec.strategy, normalizer_id=NORMALIZER_ID,
        seed=spec.seed, split='val', dataset_version=DATASET_VERSION_TAG,
        decode_config={
            'temperature': spec.temperature, 'top_p': spec.top_p,
            'max_new_tokens': MAX_NEW_TOKENS, 'do_sample': True,
            'best_of_n': spec.best_of_n, 'output_parser': 'boxed-then-last-line-v1'},
    )
    _t0    = time.perf_counter()
    result = run_baseline_eval(
        split_rows=val_rows, config=run_config,
        predictor=predictor, output_dir=RUNS_DIR / spec.run_id)
    elapsed = time.perf_counter() - _t0
    acc = float(result.summary['accuracy'])
    sparse_rows.append(dict(
        run_id=spec.run_id, strategy=spec.strategy,
        temperature=spec.temperature, top_p=spec.top_p, best_of_n=spec.best_of_n,
        seed=spec.seed, accuracy=acc, elapsed_seconds=elapsed,
        correct=int(result.summary['correct']), total=int(result.summary['total']),
    ))
    print(f'  acc={acc:.4f}  correct={result.summary["correct"]}/{result.summary["total"]}  '
          f'elapsed={elapsed:.1f}s')

print('\nSparse sweep complete.')

## Cell 14 — Aggregate sparse results

In [ ]:
sparse_aggregate = aggregate_sweep_results(
    sparse_rows, baseline_accuracy=BASELINE_ACCURACY)

sparse_df = pd.DataFrame([r.as_csv_row() for r in sparse_aggregate])
print(f'Unique sparse configs: {len(sparse_df)}')

with pd.option_context('display.max_rows', 20, 'display.float_format', '{:.4f}'.format):
    cols = ['strategy','temperature','top_p','best_of_n',
            'accuracy_mean','accuracy_std','delta_vs_baseline','significant']
    print(sparse_df[cols].to_string(index=False))

## Cell 15 — Best-of-N follow-up on the winning sparse config

In [ ]:
best_sparse = sparse_aggregate[0]
print(f'Best sparse config: strategy={best_sparse.strategy}  '
      f'temp={best_sparse.temperature}  top_p={best_sparse.top_p}  '
      f'mean_acc={best_sparse.accuracy_mean:.4f}')

bon_specs = build_best_of_n_specs(
    date_stamp=DATE_STAMP,
    strategy=best_sparse.strategy,
    temperature=best_sparse.temperature,
    top_p=best_sparse.top_p,
    seeds=DEFAULT_SEEDS,
    best_of_n_values=DEFAULT_BEST_OF_N,
)
print(f'Best-of-N runs: {len(bon_specs)}')

best_of_n_rows = []
for i, spec in enumerate(bon_specs, start=1):
    print(f'[{i:02d}/{len(bon_specs):02d}] {spec.run_id}  (BoN={spec.best_of_n})')
    predictor  = make_predictor(
        run_id=spec.run_id, strategy=spec.strategy,
        temperature=spec.temperature, top_p=spec.top_p, best_of_n=spec.best_of_n)
    run_config = make_run_config(
        run_id=spec.run_id, model_id=MODEL_ID,
        prompt_template_id=spec.strategy, normalizer_id=NORMALIZER_ID,
        seed=spec.seed, split='val', dataset_version=DATASET_VERSION_TAG,
        decode_config={
            'temperature': spec.temperature, 'top_p': spec.top_p,
            'max_new_tokens': MAX_NEW_TOKENS, 'do_sample': True,
            'best_of_n': spec.best_of_n, 'vote_rule': 'majority-v1',
            'output_parser': 'boxed-then-last-line-v1'},
    )
    _t0    = time.perf_counter()
    result = run_baseline_eval(
        split_rows=val_rows, config=run_config,
        predictor=predictor, output_dir=RUNS_DIR / spec.run_id)
    elapsed = time.perf_counter() - _t0
    acc = float(result.summary['accuracy'])
    best_of_n_rows.append(dict(
        run_id=spec.run_id, strategy=spec.strategy,
        temperature=spec.temperature, top_p=spec.top_p, best_of_n=spec.best_of_n,
        seed=spec.seed, accuracy=acc, elapsed_seconds=elapsed,
        correct=int(result.summary['correct']), total=int(result.summary['total']),
    ))
    print(f'  acc={acc:.4f}  elapsed={elapsed:.1f}s')

print('\nBest-of-N sweep complete.')

## Cell 16 — Final aggregate (sparse + Best-of-N)

In [ ]:
all_rows      = sparse_rows + best_of_n_rows
aggregate_rows = aggregate_sweep_results(all_rows, baseline_accuracy=BASELINE_ACCURACY)
aggregate_df   = pd.DataFrame([r.as_csv_row() for r in aggregate_rows])

print(f'Total unique configs: {len(aggregate_df)}')
print('\nTop 10 (ranked by mean accuracy):')
with pd.option_context('display.max_rows', 15, 'display.float_format', '{:.4f}'.format):
    cols = ['strategy','temperature','top_p','best_of_n','run_count',
            'accuracy_mean','accuracy_std','delta_vs_baseline','significant']
    print(aggregate_df[cols].head(10).to_string(index=False))

## Cell 17 — Golden regression gate

In [ ]:
PROMOTION_SEED = DEFAULT_SEEDS[0]

promoted = aggregate_rows[0]
promoted_run_id = build_run_id(
    date_stamp=DATE_STAMP, strategy=promoted.strategy,
    temperature=promoted.temperature, top_p=promoted.top_p,
    seed=PROMOTION_SEED, best_of_n=promoted.best_of_n,
) + '-golden'

print(f'Promoted config: strategy={promoted.strategy}  '
      f'temp={promoted.temperature}  top_p={promoted.top_p}  BoN={promoted.best_of_n}')
print(f'Promoted run id: {promoted_run_id}')

golden_config = make_run_config(
    run_id=promoted_run_id, model_id=MODEL_ID,
    prompt_template_id=promoted.strategy, normalizer_id=NORMALIZER_ID,
    seed=PROMOTION_SEED, split='golden',
    dataset_version=golden_rows[0].dataset_version,
    decode_config={
        'temperature': promoted.temperature, 'top_p': promoted.top_p,
        'max_new_tokens': MAX_NEW_TOKENS, 'do_sample': True,
        'best_of_n': promoted.best_of_n, 'vote_rule': 'majority-v1',
        'output_parser': 'boxed-then-last-line-v1'},
)

print(f'\nRunning golden gate on {len(golden_rows)} examples ...')
golden_predictor = make_predictor(
    run_id=promoted_run_id, strategy=promoted.strategy,
    temperature=promoted.temperature, top_p=promoted.top_p, best_of_n=promoted.best_of_n)

golden_result = run_baseline_eval(
    split_rows=golden_rows, config=golden_config,
    predictor=golden_predictor, output_dir=RUNS_DIR / promoted_run_id)

golden_gate    = evaluate_golden_gate(golden_result.records, golden_rows)
golden_summary = summarize_gate(golden_gate)

print()
print(golden_summary)
print(f'\nGolden accuracy: {float(golden_result.summary["accuracy"]):.4f}')

## Cell 18 — Write CSV and findings markdown

In [ ]:
CSV_PATH      = WORKING_DIR / f'prompting_sweep_{DATE_STAMP}.csv'
FINDINGS_PATH = WORKING_DIR / f'prompting_findings_{DATE_STAMP}.md'

aggregate_csv_path = write_aggregate_csv(aggregate_rows, CSV_PATH)

_baseline_ref = {'run_id': BASELINE_RUN_ID, 'accuracy': BASELINE_ACCURACY, 'model_id': MODEL_ID}

findings_notes = [
    f'MAX_EXAMPLES_PER_RUN={MAX_EXAMPLES_PER_RUN}; set to 200 for a full validation run.',
    'Answer extraction: prefers \\boxed{} content, falls back to last non-empty line.',
    'Baseline is greedy no-CoT (no \\boxed{} instruction) -- expected ~0% accuracy.',
    f'Model: {MODEL_ID}',
]
if not any(row.significant for row in aggregate_rows):
    findings_notes.append(
        'No config cleared the 2-sigma heuristic -- run more seeds or the full 200-example split.')

findings_md = render_findings_markdown(
    date_stamp=DATE_STAMP,
    baseline=_baseline_ref,
    aggregate_rows=aggregate_rows,
    csv_path=CSV_PATH,
    golden_gate_passed=golden_gate.passed,
    golden_summary=golden_summary,
    promoted_run_id=promoted_run_id,
    notes=findings_notes,
)
FINDINGS_PATH.write_text(findings_md, encoding='utf-8')

print(f'Wrote CSV     : {aggregate_csv_path}')
print(f'Wrote findings: {FINDINGS_PATH}')
print(f'Runs dir      : {RUNS_DIR}')

## Cell 19 — Final summary table

In [ ]:
print('=' * 72)
print('  PROMPTING SWEEP SUMMARY')
print('=' * 72)
print(f'  Date stamp           : {DATE_STAMP}')
print(f'  Model                : {MODEL_ID}')
print(f'  Max examples per run : {MAX_EXAMPLES_PER_RUN}')
print(f'  Val rows evaluated   : {len(val_rows)}')
print(f'  Baseline accuracy    : {BASELINE_ACCURACY:.4f}  ({BASELINE_ACCURACY:.1%})')
print(f'  Total seeded runs    : {len(all_rows)}')
print(f'  Unique configs       : {len(aggregate_rows)}')
print()
print(f'  Top 5 configurations (ranked by mean accuracy):')
print(f'  {"-"*70}')
print(f'  {"#":>3}  {"Strategy":<18} {"Temp":>5} {"Top-p":>6} {"BoN":>4}  '
      f'{"MeanAcc":>8}  {"Delta":>8}  {"Sig"}')
print(f'  {"-"*70}')
for rank, row in enumerate(aggregate_rows[:5], start=1):
    sig = '***' if row.significant else ''
    print(f'  {rank:>3}  {row.strategy:<18} {row.temperature:>5.2f} {row.top_p:>6.2f} '
          f'{row.best_of_n:>4}  {row.accuracy_mean:>8.4f}  {row.delta_vs_baseline:>+8.4f}  {sig}')
print(f'  {"-"*70}')
print()
gate_str = 'PASS' if golden_gate.passed else 'FAIL'
n_pass   = golden_gate.total - len(golden_gate.misses)
print(f'  Golden gate : {gate_str} ({n_pass}/{golden_gate.total} correct)')
print(f'  Promoted run: {promoted_run_id}')
print()
print(f'  Artifacts:')
print(f'    CSV     : {CSV_PATH}')
print(f'    Findings: {FINDINGS_PATH}')
print(f'    Runs    : {RUNS_DIR}')
print('=' * 72)

print('\nFull results DataFrame:')
with pd.option_context('display.max_rows', 50,
                       'display.float_format', '{:.4f}'.format,
                       'display.max_colwidth', 32):
    display(aggregate_df[['strategy','temperature','top_p','best_of_n','run_count',
                           'accuracy_mean','accuracy_std',
                           'delta_vs_baseline','significant']])